# Loan Default Risk Analytics — Internship Submission

**Project:** Loan Default Risk Analysis and Prediction  
**Model:** Random Forest Classifier (200 trees, balanced class weights)  
**Decision Threshold:** 0.40 (tuned for class imbalance)  
**Dataset:** `data/Loan_default.csv`  

---

## Table of Contents

1. [Import Libraries & Configuration](#1.-Import-Libraries-&-Configuration)
2. [Load Dataset](#2.-Load-Dataset)
3. [Data Understanding & Basic Information](#3.-Data-Understanding-&-Basic-Information)
4. [Data Quality Checks & Cleaning](#4.-Data-Quality-Checks-&-Cleaning)
5. [Exploratory Data Analysis (EDA)](#5.-Exploratory-Data-Analysis)
6. [KPI & Risk Analysis](#6.-KPI-&-Risk-Analysis)
7. [Feature Preprocessing / Engineering](#7.-Feature-Preprocessing-/-Engineering)
8. [Train the Random Forest Model](#8.-Train-the-Random-Forest-Model)
9. [Model Evaluation](#9.-Model-Evaluation)
10. [Confusion Matrix](#10.-Confusion-Matrix)
11. [Classification Report](#11.-Classification-Report)
12. [ROC-AUC](#12.-ROC-AUC)
13. [Precision-Recall / Average Precision](#13.-Precision-Recall-/-Average-Precision)
14. [Threshold Analysis (0.40)](#14.-Threshold-Analysis)
15. [Feature Importance](#15.-Feature-Importance)
16. [Example Prediction](#16.-Example-Prediction)

---

## 1. Import Libraries & Configuration

All project configuration (paths, column names, model hyper-parameters, decision threshold) is
centralised in `src/config.py`. We add the project root to `sys.path` so that the `src` package
is importable from anywhere the notebook is run.

In [ ]:
import sys
from pathlib import Path

# ── Resolve project root (works whether notebook is in root or notebooks/) ─────
NOTEBOOK_DIR = Path.cwd()
# If running from project root, use it; otherwise go one level up
if (NOTEBOOK_DIR / 'src').exists():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Standard library imports ───────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── sklearn imports ────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score,
)

# ── Project source modules ─────────────────────────────────────────────────────
from src.config import (
    RAW_DATA_PATH, TARGET_COLUMN, ID_COLUMN,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, BINARY_FEATURES,
    RANDOM_STATE, TEST_SIZE, RF_PARAMS, DECISION_THRESHOLD,
    FIGURES_DIR, TABLES_DIR,
)
from src.data_loader   import load_data, print_data_summary
from src.data_cleaner  import check_data_quality, print_quality_report, clean_data, VALID_VALUES
from src.eda import (
    plot_all_numeric_by_default, plot_categorical_default_rates,
    plot_binary_default_rates, plot_correlation_heatmap,
    get_default_rates_by_category,
)
from src.kpi           import compute_portfolio_kpis, compute_segment_kpis, print_portfolio_kpis
from src.risk_analysis import (
    assign_risk_tier, get_risk_tier_summary,
    plot_risk_tier_distribution, plot_key_drivers,
)
from src.insights      import print_insights
from src.feature_engineering import build_preprocessor, get_feature_names
from src.model import (
    train_model, evaluate_model, save_artefacts,
    plot_confusion_matrix, plot_roc_curve, plot_pr_curve,
    plot_feature_importance,
)

# ── Plot styling ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Decision threshold : {DECISION_THRESHOLD}')
print(f'Random state       : {RANDOM_STATE}')
print(f'Test size          : {TEST_SIZE}')
print('Environment ready.')

---
## 2. Load Dataset

The raw CSV is loaded from `data/Loan_default.csv` using `src.data_loader.load_data()`.
The dataset contains labelled loan applications with a binary target column `Default` (0 = no default, 1 = default).

In [ ]:
df_raw = load_data(RAW_DATA_PATH)
print(f'Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'Columns: {df_raw.columns.tolist()}')

In [ ]:
print('── First 5 rows ──')
df_raw.head()

In [ ]:
print('── Last 5 rows ──')
df_raw.tail()

---
## 3. Data Understanding & Basic Information

We inspect column data types, null counts, unique value counts, and descriptive statistics to
understand the structure of the dataset before any cleaning.

In [ ]:
# Column-level schema overview
dtype_df = pd.DataFrame({
    'Column':  df_raw.columns,
    'Dtype':   df_raw.dtypes.values.astype(str),
    'NonNull': df_raw.notnull().sum().values,
    'Null':    df_raw.isnull().sum().values,
    'Unique':  df_raw.nunique().values,
    'Sample':  [
        df_raw[c].dropna().iloc[0] if df_raw[c].dropna().shape[0] > 0 else 'N/A'
        for c in df_raw.columns
    ]
})
dtype_df

In [ ]:
# Full dataset summary via data_loader helper
print_data_summary(df_raw)

In [ ]:
# Descriptive statistics for numeric features (including skew & kurtosis)
stats = df_raw[NUMERIC_FEATURES].describe().T
stats['skewness'] = df_raw[NUMERIC_FEATURES].skew().round(3)
stats['kurtosis'] = df_raw[NUMERIC_FEATURES].kurtosis().round(3)
stats.round(2)

In [ ]:
# Target variable class distribution
class_counts = df_raw[TARGET_COLUMN].value_counts().sort_index()
class_labels = {0: 'No Default', 1: 'Default'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['#4a90d9', '#e05c5c']

axes[0].bar(
    [class_labels[k] for k in class_counts.index],
    class_counts.values,
    color=colors, edgecolor='white', linewidth=1.2
)
axes[0].set_title('Default Class Counts', fontweight='bold')
axes[0].set_ylabel('Number of Loans')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontsize=10)

axes[1].pie(
    class_counts.values,
    labels=[class_labels[k] for k in class_counts.index],
    autopct='%1.1f%%', colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Default Class Balance', fontweight='bold')

plt.suptitle('Target Variable Distribution — Loan Default', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'target_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Numeric feature histograms
n_cols = 3
n_rows = int(np.ceil(len(NUMERIC_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    axes[i].hist(df_raw[col].dropna(), bins=40, color='#4a90d9', edgecolor='white', linewidth=0.5)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for j in range(len(NUMERIC_FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'numeric_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Categorical feature bar charts
all_cat = CATEGORICAL_FEATURES + BINARY_FEATURES
n_cols = 3
n_rows = int(np.ceil(len(all_cat) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(all_cat):
    vc = df_raw[col].value_counts()
    axes[i].bar(vc.index.astype(str), vc.values, color='#7c5cd8', edgecolor='white', linewidth=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Count')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    axes[i].tick_params(axis='x', rotation=30)

for j in range(len(all_cat), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'categorical_distributions.png', bbox_inches='tight')
plt.show()

---
## 4. Data Quality Checks & Cleaning

We run a structured quality report on the raw data (missing values, duplicates, out-of-range
numeric values, invalid categorical values), then apply the cleaning pipeline from
`src.data_cleaner.clean_data()`. The cleaning pipeline:
- Drops the `LoanID` identifier column
- Removes duplicate rows
- Encodes `Yes`/`No` binary columns as `1`/`0`
- Clips numeric columns to domain-valid bounds

In [ ]:
# Print the full quality report on raw data
print_quality_report(df_raw)

In [ ]:
# Missing value percentage per column
missing_pct = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
bar_colors = ['#e05c5c' if v > 0 else '#b6d7a8' for v in missing_pct.values]
ax.bar(missing_pct.index, missing_pct.values, color=bar_colors, edgecolor='white')
ax.set_title('Missing Value Percentage by Column', fontweight='bold')
ax.set_ylabel('Missing (%)')
ax.set_ylim(0, max(missing_pct.max() + 1, 5))
ax.tick_params(axis='x', rotation=45)
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missing_values.png', bbox_inches='tight')
plt.show()

In [ ]:
# IQR-based outlier summary before cleaning
outlier_summary = []
for col in NUMERIC_FEATURES:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    lo = Q1 - 1.5 * IQR
    hi = Q3 + 1.5 * IQR
    n_out = int(((df_raw[col] < lo) | (df_raw[col] > hi)).sum())
    outlier_summary.append({
        'Feature': col,
        'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
        'Lower Fence': round(lo, 2), 'Upper Fence': round(hi, 2),
        'Outliers': n_out, 'Outlier %': round(n_out / len(df_raw) * 100, 2),
    })

outlier_df = pd.DataFrame(outlier_summary).set_index('Feature')
outlier_df

In [ ]:
# Categorical value validation against the defined valid sets
print('Categorical value validation:')
all_ok = True
for col, valid_set in VALID_VALUES.items():
    actual = set(df_raw[col].astype(str).str.strip().unique())
    unexpected = actual - valid_set
    status = 'OK' if not unexpected else f'ISSUE: {unexpected}'
    print(f'  {col:<20} {status}')
    if unexpected:
        all_ok = False
print(f'\nAll categorical values valid: {all_ok}')

In [ ]:
# Apply the cleaning pipeline
df = clean_data(df_raw)
print(f'Raw  dataset shape : {df_raw.shape}')
print(f'Clean dataset shape: {df.shape}')
print(f'Rows removed       : {len(df_raw) - len(df):,}')
print(f'Default rate       : {df[TARGET_COLUMN].mean()*100:.2f}%')
df.head()

In [ ]:
# Row count comparison — before and after
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Raw', 'After Cleaning'], [len(df_raw), len(df)],
              color=['#4a90d9', '#27ae60'], edgecolor='white', width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f"{int(bar.get_height()):,}", ha='center', fontsize=10)
ax.set_title('Row Count: Raw vs Clean Dataset', fontweight='bold')
ax.set_ylabel('Number of Rows')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'before_after_cleaning.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation matrix on clean numeric data
num_clean = df[NUMERIC_FEATURES + [TARGET_COLUMN]]
corr = num_clean.corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, linewidths=0.5,
    annot_kws={'size': 8}, ax=ax
)
ax.set_title('Correlation Matrix — Numeric Features + Target', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_matrix.png', bbox_inches='tight')
plt.show()

---
## 5. Exploratory Data Analysis

EDA is performed on the **clean** dataset. We examine how each feature relates to the default
outcome through distribution plots, default rate bar charts by category, and cross-feature
heatmaps. Key findings guide feature selection and model interpretation.

In [ ]:
# Numeric feature distributions split by default status (KDE overlays)
fig = plot_all_numeric_by_default(df)
fig.savefig(FIGURES_DIR / 'eda_numeric_by_default.png', bbox_inches='tight')
plt.show()

In [ ]:
# Absolute correlation of each numeric feature with the target
corr_target = (
    df[NUMERIC_FEATURES + [TARGET_COLUMN]]
    .corr()[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .abs()
    .sort_values(ascending=False)
    .round(4)
)
print('Absolute correlation with Default (ranked):')
print(corr_target.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
corr_target.plot.barh(ax=ax, color='#4a90d9', edgecolor='white')
ax.set_title('Absolute Correlation with Default — Numeric Features', fontweight='bold')
ax.set_xlabel('|Pearson r|')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_with_target.png', bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features — default rate per category
fig = plot_categorical_default_rates(df)
fig.savefig(FIGURES_DIR / 'eda_categorical_default_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# Binary features — default rate per flag
fig = plot_binary_default_rates(df)
fig.savefig(FIGURES_DIR / 'eda_binary_default_rates.png', bbox_inches='tight')
plt.show()

In [ ]:
# Full correlation heatmap
fig = plot_correlation_heatmap(df)
fig.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Default rate by age band
df['AgeBand'] = pd.cut(df['Age'], bins=[17, 25, 35, 45, 55, 70],
                       labels=['18-25', '26-35', '36-45', '46-55', '56-69'])
age_rates = get_default_rates_by_category(df, 'AgeBand')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(age_rates['AgeBand'].astype(str), age_rates['default_rate_pct'],
              color='#e05c5c', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Default Rate by Age Band', fontweight='bold')
ax.set_xlabel('Age Band'); ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, age_rates['default_rate_pct'].max() * 1.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_default_by_age_band.png', bbox_inches='tight')
plt.show()
df.drop(columns=['AgeBand'], inplace=True)

In [ ]:
# Default rate by income quartile
df['IncomeQuartile'] = pd.qcut(df['Income'], q=4,
                                labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])
inc_rates = get_default_rates_by_category(df, 'IncomeQuartile')

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(inc_rates['IncomeQuartile'].astype(str),
              inc_rates['default_rate_pct'], color='#7c5cd8', edgecolor='white')
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_title('Default Rate by Income Quartile', fontweight='bold')
ax.set_xlabel('Income Quartile'); ax.set_ylabel('Default Rate (%)')
ax.set_ylim(0, inc_rates['default_rate_pct'].max() * 1.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_default_by_income_quartile.png', bbox_inches='tight')
plt.show()
df.drop(columns=['IncomeQuartile'], inplace=True)

---
## 6. KPI & Risk Analysis

Portfolio-level KPIs are computed using `src.kpi`. A heuristic risk score is then assigned to
each loan via `src.risk_analysis.assign_risk_tier()`, segmenting borrowers into
**Low / Medium / High / Very High** risk tiers based on financial indicators.

In [ ]:
# Portfolio-level KPIs
print_portfolio_kpis(df)

In [ ]:
# KPI cards visualisation
kpis = compute_portfolio_kpis(df)

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
axes = axes.flatten()

kpi_items = [
    ('Total Loans',          f"{kpis['total_loans']:,}",                '#4a90d9'),
    ('Default Rate',         f"{kpis['default_rate_pct']}%",            '#e05c5c'),
    ('Total Defaulted',      f"{kpis['total_defaulted']:,}",            '#e05c5c'),
    ('Total Loan Value',     f"${kpis['total_loan_value']/1e9:.2f}B",   '#7c5cd8'),
    ('Avg Loan Amount',      f"${kpis['avg_loan_amount']:,.0f}",        '#3b82d4'),
    ('Avg Credit Score',     f"{kpis['avg_credit_score']:.0f}",         '#27ae60'),
    ('Avg Interest Rate',    f"{kpis['avg_interest_rate']}%",           '#f39c12'),
    ('Defaulted Loan Value', f"${kpis['defaulted_loan_value']/1e9:.2f}B", '#e05c5c'),
]
for ax, (label, value, color) in zip(axes, kpi_items):
    ax.set_facecolor(color)
    ax.text(0.5, 0.65, value, ha='center', va='center', fontsize=20,
            fontweight='bold', color='white', transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center', fontsize=10,
            color='white', alpha=0.9, transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.suptitle('Portfolio KPI Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'kpi_cards.png', bbox_inches='tight')
plt.show()

In [ ]:
# Segment KPIs by employment type
emp_kpis = compute_segment_kpis(df, 'EmploymentType')
print('KPIs by Employment Type:')
emp_kpis

In [ ]:
# Segment KPIs by loan purpose
purp_kpis = compute_segment_kpis(df, 'LoanPurpose')
print('KPIs by Loan Purpose:')
purp_kpis

In [ ]:
# Assign heuristic risk score and tier to each loan
df_tiers = assign_risk_tier(df)
print('Risk score range:', df_tiers['RiskScore'].min(), '—', df_tiers['RiskScore'].max())
print('\nRisk tier counts:')
print(df_tiers['RiskTier'].value_counts().to_string())

In [ ]:
# Risk tier summary with actual default rates
tier_summary = get_risk_tier_summary(df_tiers)
tier_summary.to_csv(TABLES_DIR / 'risk_tier_summary.csv', index=False)
print('Risk Tier Summary:')
tier_summary

In [ ]:
# Risk tier distribution chart
fig = plot_risk_tier_distribution(df_tiers)
fig.savefig(FIGURES_DIR / 'risk_tier_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Key drivers chart (feature-level default rate differentials)
fig = plot_key_drivers(df)
fig.savefig(FIGURES_DIR / 'key_drivers.png', bbox_inches='tight')
plt.show()

In [ ]:
# Business insights & recommendations
print_insights(df)

---
## 7. Feature Preprocessing / Engineering

A scikit-learn `ColumnTransformer` preprocessing pipeline is built via
`src.feature_engineering.build_preprocessor()`. It applies:

| Feature Group | Transformer | Columns |
|---|---|---|
| Numeric (9) | `StandardScaler` | Age, Income, LoanAmount, CreditScore, MonthsEmployed, NumCreditLines, InterestRate, LoanTerm, DTIRatio |
| Categorical (4) | `OrdinalEncoder` | Education, EmploymentType, MaritalStatus, LoanPurpose |
| Binary (3) | Passthrough (0/1) | HasMortgage, HasDependents, HasCoSigner |

The preprocessor is fitted **only on the training split** to prevent data leakage.

In [ ]:
# Build feature matrix and unfitted preprocessor
X, y, preprocessor = build_preprocessor(df)
print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'Class balance: {y.value_counts(normalize=True).round(3).to_dict()}')
print(f'Feature columns: {X.columns.tolist()}')

In [ ]:
# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f'Train set : {X_train.shape[0]:,} rows  |  Default rate: {y_train.mean()*100:.2f}%')
print(f'Test set  : {X_test.shape[0]:,} rows  |  Default rate: {y_test.mean()*100:.2f}%')

In [ ]:
# Fit preprocessor on training data only (no leakage)
preprocessor.fit(X_train)
feature_names = get_feature_names(preprocessor)
print(f'Output feature count : {len(feature_names)}')
print(f'Feature names        : {feature_names}')

In [ ]:
# Verify StandardScaler output: mean ~0, std ~1 for numeric features
X_train_transformed = preprocessor.transform(X_train)
numeric_tr = X_train_transformed[NUMERIC_FEATURES]
print('After StandardScaler — means ≈ 0, stds ≈ 1:')
pd.DataFrame({
    'Mean': numeric_tr.mean().round(4),
    'Std':  numeric_tr.std().round(4),
    'Min':  numeric_tr.min().round(3),
    'Max':  numeric_tr.max().round(3),
})

In [ ]:
# Class balance visualisation in train and test sets
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (split_name, y_split) in zip(axes, [('Train', y_train), ('Test', y_test)]):
    counts = y_split.value_counts().sort_index()
    ax.bar(['No Default', 'Default'], counts.values,
           color=['#4a90d9', '#e05c5c'], edgecolor='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)
    ax.set_title(f'{split_name} Set Class Distribution', fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('Class Balance in Train / Test Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fe_class_balance.png', bbox_inches='tight')
plt.show()

---
## 8. Train the Random Forest Model

A `RandomForestClassifier` is trained inside a full sklearn `Pipeline`
(preprocessor → classifier). Configuration from `src/config.py`:

| Parameter | Value |
|---|---|
| n_estimators | 200 |
| max_depth | None (unlimited) |
| class_weight | balanced |
| random_state | 42 |
| n_jobs | -1 (all CPU cores) |

Using `class_weight='balanced'` handles the ~11.6% default rate class imbalance.

In [ ]:
print(f'RF hyperparameters: {RF_PARAMS}')
print('Training Random Forest — this may take 1–3 minutes...')

rf_pipeline = train_model(X_train, y_train, preprocessor)

print('Training complete.')

---
## 9. Model Evaluation

The model is evaluated on the held-out test set using the project's **0.40 decision threshold**
(lower than the default 0.5 to improve recall for the minority default class). We report
accuracy, ROC-AUC, average precision, and F1 score.

In [ ]:
print(f'=== Random Forest — Evaluation on test set (threshold = {DECISION_THRESHOLD}) ===')
rf_metrics = evaluate_model(rf_pipeline, X_test, y_test, threshold=DECISION_THRESHOLD)

In [ ]:
# Summary metrics table
metrics_df = pd.DataFrame([{
    'Model':            'Random Forest',
    'Threshold':        DECISION_THRESHOLD,
    'Accuracy':         rf_metrics['accuracy'],
    'ROC-AUC':          rf_metrics['roc_auc'],
    'Avg Precision':    rf_metrics['avg_precision'],
    'F1 (Default)':     rf_metrics['f1'],
}])
metrics_df.to_csv(TABLES_DIR / 'rf_metrics_summary.csv', index=False)
print('Saved → outputs/tables/rf_metrics_summary.csv')
metrics_df

---
## 10. Confusion Matrix

The confusion matrix shows the count of true positives, true negatives, false positives and
false negatives at the chosen 0.40 decision threshold. Minimising false negatives (missed defaults)
is the primary business objective.

In [ ]:
fig = plot_confusion_matrix(rf_pipeline, X_test, y_test, threshold=DECISION_THRESHOLD)
fig.savefig(FIGURES_DIR / 'model_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved → outputs/figures/model_confusion_matrix.png')

In [ ]:
# Raw confusion matrix values
cm = rf_metrics['confusion_matrix']
tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted No Default) : {tn:,}')
print(f'False Positives (predicted Default, actually No) : {fp:,}')
print(f'False Negatives (missed Defaults)                : {fn:,}')
print(f'True Positives  (correctly predicted Default)    : {tp:,}')

---
## 11. Classification Report

Per-class precision, recall, and F1 scores. The `Default` class (minority) is the primary focus.

In [ ]:
print(f'Classification Report (threshold = {DECISION_THRESHOLD}):')
print(rf_metrics['classification_report'])

---
## 12. ROC-AUC

The ROC curve plots True Positive Rate vs False Positive Rate at every threshold.
The Area Under the Curve (AUC) measures overall discriminative ability independent of threshold.
An AUC of 1.0 is perfect; 0.5 is random.

In [ ]:
fig = plot_roc_curve(rf_pipeline, X_test, y_test)
fig.savefig(FIGURES_DIR / 'model_roc_curve.png', bbox_inches='tight')
plt.show()
print(f'ROC-AUC = {rf_metrics["roc_auc"]}')

---
## 13. Precision-Recall / Average Precision

For imbalanced classification problems the Precision-Recall curve is often more informative than
ROC. **Average Precision (AP)** summarises the area under the PR curve. The dashed baseline
corresponds to a random classifier (equal to the positive class prevalence).

In [ ]:
fig = plot_pr_curve(rf_pipeline, X_test, y_test)
fig.savefig(FIGURES_DIR / 'model_pr_curve.png', bbox_inches='tight')
plt.show()
print(f'Average Precision = {rf_metrics["avg_precision"]}')

---
## 14. Threshold Analysis

The project uses a **0.40 decision threshold** instead of the default 0.5. This improves recall
for the Default class at the cost of slightly lower precision — an acceptable trade-off because
missing a true default (false negative) is more costly than a false alarm (false positive).

The plot below shows how Precision, Recall, and F1 vary across thresholds from 0.10 to 0.75.

In [ ]:
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.10, 0.80, 0.05)

rows = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    rows.append({
        'Threshold': round(t, 2),
        'Precision': round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        'Recall':    round(recall_score(y_test, y_pred_t, zero_division=0), 4),
        'F1':        round(f1_score(y_test, y_pred_t, zero_division=0), 4),
    })

thresh_df = pd.DataFrame(rows)
thresh_df.to_csv(TABLES_DIR / 'threshold_analysis.csv', index=False)
print('Saved → outputs/tables/threshold_analysis.csv')
thresh_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'], label='Precision',
        color='#4a90d9', marker='o', markersize=4)
ax.plot(thresh_df['Threshold'], thresh_df['Recall'],    label='Recall',
        color='#e05c5c', marker='o', markersize=4)
ax.plot(thresh_df['Threshold'], thresh_df['F1'],        label='F1 Score',
        color='#27ae60', marker='o', markersize=4)
ax.axvline(DECISION_THRESHOLD, color='k', linestyle='--', linewidth=1.5,
           label=f'Chosen threshold ({DECISION_THRESHOLD})')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Decision Threshold', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_threshold_analysis.png', bbox_inches='tight')
plt.show()

In [ ]:
# Predicted probability distribution — Default vs No Default
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_prob[y_test == 0], bins=50, alpha=0.55, color='#4a90d9',
        label='No Default', density=True, edgecolor='none')
ax.hist(y_prob[y_test == 1], bins=50, alpha=0.55, color='#e05c5c',
        label='Default', density=True, edgecolor='none')
ax.axvline(DECISION_THRESHOLD, color='k', linestyle='--', linewidth=1.5,
           label=f'Threshold = {DECISION_THRESHOLD}')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Density')
ax.set_title('Predicted Probability Distribution by Actual Label', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_prob_distribution.png', bbox_inches='tight')
plt.show()

---
## 15. Feature Importance

Random Forest provides mean decrease in Gini impurity for each feature. Higher importance means
the feature contributes more to reducing impurity (i.e., better separates defaulters from
non-defaulters) across all 200 trees.

In [ ]:
fig = plot_feature_importance(rf_pipeline, top_n=16)
fig.savefig(FIGURES_DIR / 'model_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance as a ranked table
clf = rf_pipeline.named_steps['classifier']
prep = rf_pipeline.named_steps['preprocessor']
feat_names_out = get_feature_names(prep)

importance_df = pd.DataFrame({
    'Feature':    feat_names_out,
    'Importance': clf.feature_importances_,
}).sort_values('Importance', ascending=False).reset_index(drop=True)
importance_df['Importance'] = importance_df['Importance'].round(4)
importance_df.to_csv(TABLES_DIR / 'feature_importances.csv', index=False)
print('Saved → outputs/tables/feature_importances.csv')
importance_df

---
## 16. Example Prediction

We demonstrate live inference using `src.model.predict_default()` on three samples from the
test set. The function returns:
- **probability** — model's estimated P(default)
- **prediction** — binary label at the 0.40 threshold  
- **risk_label** — human-readable tier: Low / Medium / High / Very High

Risk label mapping:
| Probability range | Risk Label |
|---|---|
| ≥ 0.60 | Very High |
| 0.40 – 0.59 | High |
| 0.20 – 0.39 | Medium |
| < 0.20 | Low |

In [ ]:
from src.model import predict_default

# Select 3 test samples (1 known default, 1 known non-default, 1 borderline)
default_idx    = y_test[y_test == 1].index[:1]
no_default_idx = y_test[y_test == 0].index[:1]
# Find a borderline sample (probability closest to threshold)
border_prob = np.abs(y_prob - DECISION_THRESHOLD)
border_idx  = [X_test.index[np.argmin(border_prob)]]

sample_idx = default_idx.tolist() + no_default_idx.tolist() + border_idx
sample = X_test.loc[sample_idx].copy()

result = predict_default(rf_pipeline, sample, threshold=DECISION_THRESHOLD)

out = sample[['CreditScore', 'InterestRate', 'DTIRatio', 'Income', 'LoanAmount']].copy()
out['Actual']      = y_test.loc[sample_idx].values
out['Probability'] = [round(p, 4) for p in result['probability']]
out['Prediction']  = result['prediction']
out['RiskLabel']   = result['risk_label']
out

In [ ]:
# Demonstrate prediction on a fully manually constructed input
manual_input = pd.DataFrame([{
    'Age':             35,
    'Income':          50000,
    'LoanAmount':      20000,
    'CreditScore':     620,
    'MonthsEmployed':  24,
    'NumCreditLines':  3,
    'InterestRate':    12.5,
    'LoanTerm':        36,
    'DTIRatio':        0.45,
    'Education':       'Bachelor\'s',
    'EmploymentType':  'Full-time',
    'MaritalStatus':   'Single',
    'LoanPurpose':     'Auto',
    'HasMortgage':     0,
    'HasDependents':   1,
    'HasCoSigner':     0,
}])

manual_result = predict_default(rf_pipeline, manual_input, threshold=DECISION_THRESHOLD)
print('Manual Input Prediction:')
print(f"  Default Probability : {manual_result['probability'][0]:.4f}")
print(f"  Prediction (0/1)    : {manual_result['prediction'][0]}")
print(f"  Risk Label          : {manual_result['risk_label'][0]}")

---
## Save Model Artefacts

The fitted pipeline, preprocessor, and feature names are saved to `models/` so the Streamlit
dashboard can load them for live inference without re-training.

In [ ]:
save_artefacts(rf_pipeline, preprocessor)
print('Model artefacts saved to models/')

---
## Project Summary

| Section | Key Result |
|---|---|
| Dataset | 255,347 loans × 18 columns (after cleaning) |
| Default rate | ~11.6% (class imbalance present) |
| Features used | 16 (9 numeric + 4 categorical + 3 binary) |
| Model | Random Forest (200 trees, balanced weights) |
| Decision threshold | 0.40 (tuned for recall on Default class) |
| ROC-AUC | See cell 12 output above |
| Average Precision | See cell 13 output above |
| F1 (Default class) | See cell 9 output above |
| Model artefacts | `models/random_forest_model.joblib` |
| Dashboard | `dashboard/` — run with `streamlit run dashboard/app.py` |

---
*Loan Default Risk Analytics — Internship Submission*